In [ ]:
!pip install fastapi uvicorn nest-asyncio pyngrok openai -q
!pip install langchain langchain-community langchain-openai -q
!pip install chromadb pypdf python-multipart tiktoken easyocr pillow opencv-python -q

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import uvicorn
import openai
import easyocr
import numpy as np

from io import BytesIO
from PIL import Image

from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware

from pydantic import BaseModel

from pypdf import PdfReader

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

from pyngrok import ngrok

/tmp/ipykernel_52055/3405397060.py:21: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [ ]:
app = FastAPI()


app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"]
)

In [ ]:
client=openai.OpenAI(
api_key="sk-EF5sUlx6Li_oUp0N5E6R5Q",
base_url="https://apidev.navigatelabsai.com"
)

In [ ]:
ocr_reader=easyocr.Reader(["en"])


splitter=RecursiveCharacterTextSplitter(
chunk_size=1000,
chunk_overlap=200
)


embeddings=OpenAIEmbeddings(
model="text-embedding-3-small",
api_key="sk-EF5sUlx6Li_oUp0N5E6R5Q",
base_url="https://apidev.navigatelabsai.com"
)


vector_db=None
retriever=None

In [ ]:
class PromptRequest(BaseModel):
    question:str


class QuizRequest(BaseModel):
    topic:str
    difficulty:str="medium"
    questions:int=10


class StudyPlanRequest(BaseModel):
    subjects:list
    exam_days:int
    hours_per_day:int


class ProgressRequest(BaseModel):
    scores:dict


class VoiceRequest(BaseModel):
    transcript:str

In [ ]:
@app.post("/upload_study_material/")
async def upload_study_material(
    file: UploadFile = File(...)
):

    global vector_db
    global retriever

    try:

        print("UPLOAD STARTED")

        reader = PdfReader(file.file)

        text = ""

        for page in reader.pages:
            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"


        if not text.strip():
            return {
                "error":"No text found in PDF"
            }


        chunks = text_splitter.split_text(text)


        vector_db = Chroma.from_texts(
            chunks,
            embeddings
        )


        retriever = vector_db.as_retriever(
            search_kwargs={"k":5}
        )


        print("UPLOAD SUCCESS")


        return {
            "status":"success",
            "chunks":len(chunks)
        }


    except Exception as e:

        print("UPLOAD ERROR:",e)

        return {
            "error":str(e)
        }

In [ ]:
@app.post("/ask")
async def ask(req:PromptRequest):

    if retriever is None:
        return {"error":"Upload PDF first"}


    docs=retriever.invoke(req.question)


    context="\n".join(
    [d.page_content for d in docs]
    )


    response=client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
    {
    "role":"user",
    "content":
    f"Context:{context}\nQuestion:{req.question}"
    }
    ])

    return {
    "answer":
    response.choices[0].message.content
    }

In [ ]:
@app.post("/voice_tutor")
async def voice_tutor(
    req: VoiceRequest
):

    global retriever

    try:

        if retriever is None:
            return {
                "error": "Please upload study material first"
            }


        docs = retriever.invoke(
            req.transcript
        )


        context = "\n".join(
            [doc.page_content for doc in docs]
        )


        prompt = f"""
You are an AI Tutor.

Use this context:
{context}

Student question:
{req.transcript}

Explain in simple language.
"""


        response = client.chat.completions.create(
            model="gemini-2.5-flash",
            messages=[
                {
                    "role":"user",
                    "content":prompt
                }
            ]
        )


        return {
            "answer":
            response.choices[0].message.content
        }


    except Exception as e:

        print("VOICE ERROR:", e)

        return {
            "error":str(e)
        }

In [ ]:
@app.post("/summary")
async def generate_summary():
    global retriever

    try:
        if retriever is None:
            return {"error": "Please upload study material first"}

        docs = retriever.get_relevant_documents("important topics")

        if not docs:
            return {"error": "No documents found in uploaded material"}

        context = "\n\n".join(
            doc.page_content for doc in docs if doc.page_content
        )

        if not context.strip():
            return {"error": "Uploaded material contained no readable text"}

        response = client.chat.completions.create(
            model="gemini-2.5-flash",
            messages=[
                {
                    "role": "user",
                    "content": f"""
Generate detailed study notes.

Context:
{context}
"""
                }
            ]
        )

        return {
            "summary": response.choices[0].message.content
        }

    except Exception as e:
        print("SUMMARY ERROR:", e)
        return {"error": str(e)}

In [ ]:
@app.post("/quiz")
async def generate_quiz(
    req: QuizRequest
):

    prompt = f"""
Create {req.questions} MCQs.

Topic:
{req.topic}

Difficulty:
{req.difficulty}

Include answers.
"""

    response = client.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ]
    )

    return {
        "quiz":
        response.choices[0].message.content
    }

In [ ]:
@app.post("/study_plan")
async def study_plan(
    req: StudyPlanRequest
):

    prompt = f"""
Create study plan.

Subjects:
{req.subjects}

Days:
{req.exam_days}

Hours:
{req.hours_per_day}
"""

    response = client.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ]
    )

    return {
        "plan":
        response.choices[0].message.content
    }

In [ ]:
@app.post("/analyze_progress")
async def analyze_progress(
    req: ProgressRequest
):

    prompt = f"""
Analyze student scores.

Scores:

{req.scores}

Recommend weak topics.
"""

    response = client.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ]
    )

    return {
        "analysis":
        response.choices[0].message.content
    }

In [ ]:
@app.get("/")
def root():

    return {
        "message":"AI Study Assistant Running"
    }

In [ ]:
import os
import subprocess

# Kill the process listening on port 8000 (FastAPI app)
try:
    # First, get the PIDs listening on port 8000
    pid_command = "lsof -t -i:8000"
    # Use check=False so subprocess.run doesn't raise an error if lsof finds no processes
    pid_result = subprocess.run(pid_command, shell=True, check=False, capture_output=True, text=True)

    if pid_result.returncode == 0 and pid_result.stdout.strip():
        pids = pid_result.stdout.strip().split()
        for pid in pids:
            try:
                subprocess.run(['kill', '-9', pid], check=True, capture_output=True, text=True)
                print(f"Process {pid} on port 8000 killed.")
            except subprocess.CalledProcessError as e:
                print(f"Error killing process {pid} on port 8000: {e.stderr.strip()}")
        print("All processes on port 8000 killed.")
    elif pid_result.returncode != 0 and pid_result.stderr.strip():
        print(f"Error running lsof: {pid_result.stderr.strip()}")
    else:
        print("No process found listening on port 8000.")

except Exception as e:
    print(f"An unexpected error occurred while trying to kill process on port 8000: {e}")

# Kill all ngrok tunnels
try:
    # Ensure ngrok is imported for kill()
    from pyngrok import ngrok
    ngrok.kill()
    print("All ngrok tunnels killed.")
except ImportError:
    print("pyngrok module not found. Cannot kill ngrok tunnels gracefully.")
except Exception as e:
    print(f"Error killing ngrok tunnels: {e}. An existing ngrok process might still be running externally.")
    # Fallback to killall ngrok if ngrok.kill() fails or module is not fully initialized
    try:
        subprocess.run(['killall', 'ngrok'], check=True, capture_output=True, shell=True, text=True)
        print("Attempted to kill ngrok processes via killall as a fallback.")
    except subprocess.CalledProcessError as e:
        print(f"No ngrok processes found to kill via killall or error during killall: {e.stderr.strip()}")
    except Exception as e_fallback:
        print(f"An unexpected error occurred trying to kill ngrok via killall: {e_fallback}")

No process found listening on port 8000.
All ngrok tunnels killed.


In [ ]:
import asyncio

async def start_fastapi():

    config = uvicorn.Config(
        app=app,
        host="0.0.0.0",
        port=8000
    )

    server = uvicorn.Server(config)

    await server.serve()

asyncio.get_event_loop().create_task(
    start_fastapi()
)

<Task pending name='Task-1' coro=<start_fastapi() running at /tmp/ipykernel_52055/1692278228.py:3>>

In [ ]:
!ngrok config add-authtoken "3Dz0G8HAcq9xAirMapSLlOdjr9z_2A7NUkh4gXFZGs6Y9P5g5"

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
ngrok.kill()


public_url=ngrok.connect(8000)

print(
"Backend URL:",
public_url.public_url
)


Backend URL: https://astronaut-recall-bobcat.ngrok-free.dev
